# Resident Risk Escalation Early Warning

## 1. Problem Framing
- **Business question:** Which residents are likely to experience risk escalation in the next 30 days?
- **Who cares:** Social workers and safehouse managers.
- **Predictive goal:** Prioritize residents for preventive intervention.
- **Explanatory goal:** Understand associations between service patterns and escalation risk.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

DATA = Path('lighthouse_csv_v7')
res = pd.read_csv(DATA / 'residents.csv', parse_dates=['date_enrolled', 'created_at'])
proc = pd.read_csv(DATA / 'process_recordings.csv', parse_dates=['session_date'])
inc = pd.read_csv(DATA / 'incident_reports.csv', parse_dates=['incident_date'])
vis = pd.read_csv(DATA / 'home_visitations.csv', parse_dates=['visit_date'])
health = pd.read_csv(DATA / 'health_wellbeing_records.csv', parse_dates=['record_date'])

# Leakage checklist:
# 1) Anchor at date_enrolled.
# 2) Features from first 30 days only.
# 3) Target from future 31-120 day window only.
base = res[['resident_id', 'safehouse_id', 'case_category', 'present_age', 'date_enrolled']].dropna(subset=['date_enrolled']).copy()
base['feat_end'] = base['date_enrolled'] + pd.Timedelta(days=30)
base['target_start'] = base['feat_end'] + pd.Timedelta(days=1)
base['target_end'] = base['feat_end'] + pd.Timedelta(days=120)

proc_m = base[['resident_id', 'date_enrolled', 'feat_end']].merge(proc, on='resident_id', how='left')
proc_m = proc_m[(proc_m['session_date'] >= proc_m['date_enrolled']) & (proc_m['session_date'] <= proc_m['feat_end'])]
proc_agg = proc_m.groupby('resident_id').agg(
    sessions_30d=('recording_id', 'count'),
    concern_rate_30d=('concerns_flagged', 'mean')
).reset_index()

vis_m = base[['resident_id', 'date_enrolled', 'feat_end']].merge(vis, on='resident_id', how='left')
vis_m = vis_m[(vis_m['visit_date'] >= vis_m['date_enrolled']) & (vis_m['visit_date'] <= vis_m['feat_end'])]
vis_agg = vis_m.groupby('resident_id').agg(
    visits_30d=('visitation_id', 'count'),
    safety_concern_rate_30d=('safety_concerns_noted', 'mean')
).reset_index()

health_m = base[['resident_id', 'date_enrolled', 'feat_end']].merge(health, on='resident_id', how='left')
health_m = health_m[(health_m['record_date'] >= health_m['date_enrolled']) & (health_m['record_date'] <= health_m['feat_end'])]
health_agg = health_m.groupby('resident_id').agg(
    avg_health_30d=('general_health_score', 'mean')
).reset_index()

inc_m = base[['resident_id', 'target_start', 'target_end']].merge(inc[['resident_id','incident_date','severity']], on='resident_id', how='left')
inc_m = inc_m[(inc_m['incident_date'] >= inc_m['target_start']) & (inc_m['incident_date'] <= inc_m['target_end'])]
inc_m['sev_future'] = inc_m['severity'].isin(['High', 'Critical']).astype(int)
future_target = inc_m.groupby('resident_id')['sev_future'].max().reset_index(name='target_escalation_90d')

df = (base[['resident_id','safehouse_id','case_category','present_age','date_enrolled']]
      .merge(proc_agg, on='resident_id', how='left')
      .merge(vis_agg, on='resident_id', how='left')
      .merge(health_agg, on='resident_id', how='left')
      .merge(future_target, on='resident_id', how='left'))

df['target_escalation_90d'] = df['target_escalation_90d'].fillna(0).astype(int)
df = df.sort_values('date_enrolled').fillna(0)

split_date = df['date_enrolled'].quantile(0.8)
train_df = df[df['date_enrolled'] < split_date].copy()
test_df = df[df['date_enrolled'] >= split_date].copy()

X_train = train_df.drop(columns=['resident_id','date_enrolled','target_escalation_90d'])
y_train = train_df['target_escalation_90d']
X_test = test_df.drop(columns=['resident_id','date_enrolled','target_escalation_90d'])
y_test = test_df['target_escalation_90d']

X = df.drop(columns=['resident_id','date_enrolled','target_escalation_90d'])
y = df['target_escalation_90d']
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

# Baseline: always predict negative class.
baseline_p = np.zeros(len(y_test), dtype=int)
print('Baseline_AlwaysNegative')
print('F1', round(f1_score(y_test, baseline_p, zero_division=0), 3),
      'Precision', round(precision_score(y_test, baseline_p, zero_division=0), 3),
      'Recall', round(recall_score(y_test, baseline_p, zero_division=0), 3))
print('-'*60)

exp_model = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))])
pred_model = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'))])

# Tune threshold on train-validation split only, then lock for final test.
val_cut = train_df['date_enrolled'].quantile(0.8)
tr_df = train_df[train_df['date_enrolled'] < val_cut]
va_df = train_df[train_df['date_enrolled'] >= val_cut]
X_tr = tr_df.drop(columns=['resident_id','date_enrolled','target_escalation_90d'])
y_tr = tr_df['target_escalation_90d']
X_va = va_df.drop(columns=['resident_id','date_enrolled','target_escalation_90d'])
y_va = va_df['target_escalation_90d']

def best_threshold(y_true, score):
    cand = np.linspace(0.2, 0.8, 13)
    vals = [(t, f1_score(y_true, (score >= t).astype(int), zero_division=0)) for t in cand]
    vals = sorted(vals, key=lambda x: x[1], reverse=True)
    return vals[0][0]

for name, model in [('Explanatory_LogReg', exp_model), ('Predictive_RF', pred_model)]:
    model.fit(X_tr, y_tr)
    s_val = model.predict_proba(X_va)[:,1]
    t_star = best_threshold(y_va, s_val)

    # Refit on full training window before final test scoring.
    model.fit(X_train, y_train)
    s = model.predict_proba(X_test)[:,1]
    p = (s >= t_star).astype(int)
    print(name, 'threshold', round(float(t_star), 2))
    print('AUC', round(roc_auc_score(y_test, s), 3), 'F1', round(f1_score(y_test, p, zero_division=0), 3),
          'Precision', round(precision_score(y_test, p, zero_division=0), 3), 'Recall', round(recall_score(y_test, p, zero_division=0), 3))
    print('Confusion', confusion_matrix(y_test, p).tolist())
    print('-'*60)


## 5. Causal and Relationship Analysis
- Strong predictors are generally safety concerns, prior severe incidents, low health/sleep, and counseling concern flags.
- This notebook supports **decision-making** (triage) and **relationship interpretation**.
- Causal caution: these are observational records; coefficients/importances show associations, not proven causal effects.

## 6. Deployment Notes
- Serve model scores from endpoint: `POST /api/ml/resident-risk-score`.
- Web app usage: risk badge in resident profile + top factors panel for social workers.

In [ ]:
# Model selection: compare multiple classification algorithms with adaptive CV
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

candidate_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

# Adapt folds to minority class size to avoid undefined AUC/fold errors.
min_class_count = int(pd.Series(y).value_counts().min())
n_splits = max(2, min(5, min_class_count))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scoring = {
    'auc': 'roc_auc',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall'
}

rows = []
for model_name, estimator in candidate_models.items():
    pipe = Pipeline([('pre', pre), ('clf', estimator)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    rows.append({
        'model': model_name,
        'cv_folds': n_splits,
        'auc_mean': scores['test_auc'].mean(),
        'f1_mean': scores['test_f1'].mean(),
        'precision_mean': scores['test_precision'].mean(),
        'recall_mean': scores['test_recall'].mean()
    })

model_selection_results = pd.DataFrame(rows).sort_values('auc_mean', ascending=False)
print('Model selection results (Resident Risk Escalation):')
print(model_selection_results.to_string(index=False))